In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, json, math

# always start fresh from disk
matches_df = pd.read_csv("../data/processed/matches.csv")
matches_df["date"] = pd.to_datetime(matches_df["date"])
deliveries_df = pd.read_csv("../data/processed/deliveries.csv")

print("matches_df:", matches_df.shape)
print("deliveries_df:", deliveries_df.shape)
print("Columns OK:", "has_impact_player_rule" in matches_df.columns and "phase" in deliveries_df.columns)

matches_df: (1243, 42)
deliveries_df: (295732, 21)
Columns OK: True


In [2]:
# build a match-by-match record for each team
# each match appears twice — once as bat_first_team, once as bat_second_team
team_match_log = pd.concat([
    matches_df[["match_id","date","bat_first_team","bat_second_team",
                "total_runs_inn1","total_runs_inn2","winner"]].rename(
        columns={"bat_first_team":"team","bat_second_team":"opponent",
                 "total_runs_inn1":"runs_scored","total_runs_inn2":"runs_conceded"}),
    matches_df[["match_id","date","bat_second_team","bat_first_team",
                "total_runs_inn2","total_runs_inn1","winner"]].rename(
        columns={"bat_second_team":"team","bat_first_team":"opponent",
                 "total_runs_inn2":"runs_scored","total_runs_inn1":"runs_conceded"})
], ignore_index=True)

team_match_log["won"] = (team_match_log["team"] == team_match_log["winner"]).astype(int)
team_match_log = team_match_log.sort_values("date").reset_index(drop=True)

team_match_log.head(10)

,match_id,date,team,opponent,runs_scored,runs_conceded,winner,won
0,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bengaluru,222.0,82.0,Kolkata Knight Riders,1
1,335982,2008-04-18,Royal Challengers Bengaluru,Kolkata Knight Riders,82.0,222.0,Kolkata Knight Riders,0
2,335983,2008-04-19,Chennai Super Kings,Punjab Kings,240.0,207.0,Chennai Super Kings,1
3,335984,2008-04-19,Rajasthan Royals,Delhi Capitals,129.0,132.0,Delhi Capitals,0
4,335984,2008-04-19,Delhi Capitals,Rajasthan Royals,132.0,129.0,Delhi Capitals,1
5,335983,2008-04-19,Punjab Kings,Chennai Super Kings,207.0,240.0,Chennai Super Kings,0
6,335986,2008-04-20,Kolkata Knight Riders,Sunrisers Hyderabad,112.0,110.0,Kolkata Knight Riders,1
7,335985,2008-04-20,Royal Challengers Bengaluru,Mumbai Indians,166.0,165.0,Royal Challengers Bengaluru,1
8,335986,2008-04-20,Sunrisers Hyderabad,Kolkata Knight Riders,110.0,112.0,Kolkata Knight Riders,0
9,335985,2008-04-20,Mumbai Indians,Royal Challengers Bengaluru,165.0,166.0,Royal Challengers Bengaluru,0


In [3]:
# sort by team + date so rolling window is chronological per team
team_match_log = team_match_log.sort_values(["team", "date"]).reset_index(drop=True)

# rolling 5-match averages, shifted by 1 so we don't include the current match
team_match_log["form_runs_scored"] = (
    team_match_log.groupby("team")["runs_scored"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .round(1)
)

team_match_log["form_runs_conceded"] = (
    team_match_log.groupby("team")["runs_conceded"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .round(1)
)

team_match_log["form_win_rate"] = (
    team_match_log.groupby("team")["won"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .round(3)
)

team_match_log[["match_id","team","date","runs_scored","form_runs_scored","form_runs_conceded","form_win_rate"]].head(20)

,match_id,team,date,runs_scored,form_runs_scored,form_runs_conceded,form_win_rate
0,335983,Chennai Super Kings,2008-04-19,240.0,NaN,NaN,NaN
1,335989,Chennai Super Kings,2008-04-23,208.0,240.0,207.0,1.0
2,335993,Chennai Super Kings,2008-04-26,152.0,224.0,204.5,1.0
3,335996,Chennai Super Kings,2008-04-28,178.0,200.0,185.3,1.0
4,336001,Chennai Super Kings,2008-05-02,169.0,194.5,180.2,1.0
5,336005,Chennai Super Kings,2008-05-04,109.0,189.4,178.6,0.8
6,336007,Chennai Super Kings,2008-05-06,144.0,163.2,159.2,0.6
7,336009,Chennai Super Kings,2008-05-08,188.0,150.4,148.4,0.4
8,336013,Chennai Super Kings,2008-05-10,181.0,157.6,156.4,0.4
9,336018,Chennai Super Kings,2008-05-14,156.0,158.2,156.0,0.4


In [4]:
# get form stats for bat_first_team
form_cols = ["match_id", "team", "form_runs_scored", "form_runs_conceded", "form_win_rate"]

bat_first_form = team_match_log[form_cols].rename(columns={
    "team": "bat_first_team",
    "form_runs_scored": "team1_form_runs_scored",
    "form_runs_conceded": "team1_form_runs_conceded",
    "form_win_rate": "team1_form_win_rate"
})

bat_second_form = team_match_log[form_cols].rename(columns={
    "team": "bat_second_team",
    "form_runs_scored": "team2_form_runs_scored",
    "form_runs_conceded": "team2_form_runs_conceded",
    "form_win_rate": "team2_form_win_rate"
})

matches_df = matches_df.merge(bat_first_form, on=["match_id", "bat_first_team"], how="left")
matches_df = matches_df.merge(bat_second_form, on=["match_id", "bat_second_team"], how="left")

matches_df[["match_id", "bat_first_team", "bat_second_team",
            "team1_form_runs_scored", "team1_form_win_rate",
            "team2_form_runs_scored", "team2_form_win_rate"]].head(10)

,match_id,bat_first_team,bat_second_team,team1_form_runs_scored,team1_form_win_rate,team2_form_runs_scored,team2_form_win_rate
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,NaN,NaN,NaN,NaN
1,335983,Chennai Super Kings,Punjab Kings,NaN,NaN,NaN,NaN
2,335984,Rajasthan Royals,Delhi Capitals,NaN,NaN,NaN,NaN
3,335986,Sunrisers Hyderabad,Kolkata Knight Riders,NaN,NaN,222.0,1.0
4,335985,Mumbai Indians,Royal Challengers Bengaluru,NaN,NaN,82.0,0.0
5,335987,Punjab Kings,Rajasthan Royals,207.0,0.0,129.0,0.0
6,335988,Sunrisers Hyderabad,Delhi Capitals,110.0,0.0,132.0,1.0
7,335989,Chennai Super Kings,Mumbai Indians,240.0,1.0,165.0,0.0
8,335990,Sunrisers Hyderabad,Rajasthan Royals,126.0,0.0,148.5,0.5
9,335991,Punjab Kings,Mumbai Indians,186.5,0.0,183.5,0.0


In [5]:
# build head to head record
h2h_log = team_match_log[["match_id", "date", "team", "opponent", "won"]].copy()

h2h_log = h2h_log.sort_values(["team", "opponent", "date"]).reset_index(drop=True)

h2h_log["h2h_wins"] = (
    h2h_log.groupby(["team", "opponent"])["won"]
    .transform(lambda x: x.shift(1).expanding().sum())
)

h2h_log["h2h_matches"] = (
    h2h_log.groupby(["team", "opponent"])["won"]
    .transform(lambda x: x.shift(1).expanding().count())
)

h2h_log["h2h_win_rate"] = (h2h_log["h2h_wins"] / h2h_log["h2h_matches"]).round(3)

# merge onto matches_df for bat_first_team vs bat_second_team
h2h_cols = ["match_id", "team", "opponent", "h2h_win_rate", "h2h_matches"]

bat_first_h2h = h2h_log[h2h_log["match_id"].isin(matches_df["match_id"])][h2h_cols].rename(columns={
    "team": "bat_first_team",
    "opponent": "bat_second_team",
    "h2h_win_rate": "team1_h2h_win_rate",
    "h2h_matches": "team1_h2h_matches"
})

matches_df = matches_df.merge(bat_first_h2h, on=["match_id", "bat_first_team", "bat_second_team"], how="left")

matches_df[["match_id", "bat_first_team", "bat_second_team", "team1_h2h_win_rate", "team1_h2h_matches"]].head(15)

,match_id,bat_first_team,bat_second_team,team1_h2h_win_rate,team1_h2h_matches
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,NaN,0.0
1,335983,Chennai Super Kings,Punjab Kings,NaN,0.0
2,335984,Rajasthan Royals,Delhi Capitals,NaN,0.0
3,335986,Sunrisers Hyderabad,Kolkata Knight Riders,NaN,0.0
4,335985,Mumbai Indians,Royal Challengers Bengaluru,NaN,0.0
5,335987,Punjab Kings,Rajasthan Royals,NaN,0.0
6,335988,Sunrisers Hyderabad,Delhi Capitals,NaN,0.0
7,335989,Chennai Super Kings,Mumbai Indians,NaN,0.0
8,335990,Sunrisers Hyderabad,Rajasthan Royals,NaN,0.0
9,335991,Punjab Kings,Mumbai Indians,NaN,0.0


In [6]:
matches_df[["match_id", "bat_first_team", "bat_second_team", 
            "team1_h2h_win_rate", "team1_h2h_matches"]].dropna().head(10)

,match_id,bat_first_team,bat_second_team,team1_h2h_win_rate,team1_h2h_matches
27,336009,Delhi Capitals,Chennai Super Kings,1.0,1.0
28,336010,Kolkata Knight Riders,Royal Challengers Bengaluru,1.0,1.0
29,336011,Sunrisers Hyderabad,Rajasthan Royals,0.0,1.0
30,336013,Chennai Super Kings,Punjab Kings,1.0,1.0
31,336014,Kolkata Knight Riders,Sunrisers Hyderabad,1.0,1.0
32,336015,Delhi Capitals,Rajasthan Royals,1.0,1.0
33,336016,Royal Challengers Bengaluru,Punjab Kings,0.0,1.0
35,336018,Chennai Super Kings,Mumbai Indians,1.0,1.0
36,336020,Delhi Capitals,Sunrisers Hyderabad,1.0,1.0
37,336021,Kolkata Knight Riders,Mumbai Indians,0.0,1.0


In [7]:
# for each match, what was the average first innings score at this venue
# in ALL prior matches at that venue (before this match's date)
matches_sorted = matches_df.sort_values("date").reset_index(drop=True)

venue_avg = []
for idx, row in matches_sorted.iterrows():
    prior = matches_sorted[
        (matches_sorted["venue"] == row["venue"]) &
        (matches_sorted["date"] < row["date"])
    ]
    if len(prior) >= 3:  # need at least 3 prior matches for a meaningful average
        venue_avg.append(prior["total_runs_inn1"].mean().round(1))
    else:
        venue_avg.append(np.nan)

matches_sorted["venue_avg_score"] = venue_avg
matches_df = matches_sorted.copy()

matches_df[["match_id", "venue", "total_runs_inn1", "venue_avg_score"]].dropna().head(10)

,match_id,venue,total_runs_inn1,venue_avg_score
20,336003,"Punjab Cricket Association Stadium, Mohali",178,193.3
21,336034,M Chinnaswamy Stadium,156,178.3
24,336006,M Chinnaswamy Stadium,126,172.8
25,336007,MA Chidambaram Stadium,144,174.7
29,336011,Sawai Mansingh Stadium,140,157.0
30,336013,MA Chidambaram Stadium,181,167.0
31,336014,Rajiv Gandhi International Stadium,204,173.3
32,336015,Sawai Mansingh Stadium,156,152.8
33,336016,"Punjab Cricket Association Stadium, Mohali",143,189.5
34,336017,Eden Gardens,133,125.3


In [8]:
# for each venue, what % of matches were won by the toss winner historically
decided = matches_df[matches_df["result"] == "normal"].copy()
decided["toss_winner_won"] = (decided["toss_winner"] == decided["winner"]).astype(int)

# expanding historical toss win rate per venue, before each match
matches_sorted2 = decided.sort_values("date").reset_index(drop=True)

toss_advantage = []
for idx, row in matches_sorted2.iterrows():
    prior = matches_sorted2[
        (matches_sorted2["venue"] == row["venue"]) &
        (matches_sorted2["date"] < row["date"])
    ]
    if len(prior) >= 3:
        toss_advantage.append(prior["toss_winner_won"].mean().round(3))
    else:
        toss_advantage.append(np.nan)

matches_sorted2["venue_toss_win_rate"] = toss_advantage

# merge back onto matches_df
matches_df = matches_df.merge(
    matches_sorted2[["match_id", "venue_toss_win_rate"]],
    on="match_id", how="left"
)

matches_df[["match_id", "venue", "toss_winner", "winner", "venue_toss_win_rate"]].dropna().head(10)

,match_id,venue,toss_winner,winner,venue_toss_win_rate
20,336003,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Punjab Kings,0.333
21,336034,M Chinnaswamy Stadium,Sunrisers Hyderabad,Royal Challengers Bengaluru,0.667
24,336006,M Chinnaswamy Stadium,Punjab Kings,Punjab Kings,0.500
25,336007,MA Chidambaram Stadium,Sunrisers Hyderabad,Sunrisers Hyderabad,0.000
29,336011,Sawai Mansingh Stadium,Rajasthan Royals,Rajasthan Royals,0.333
30,336013,MA Chidambaram Stadium,Punjab Kings,Chennai Super Kings,0.250
31,336014,Rajiv Gandhi International Stadium,Kolkata Knight Riders,Kolkata Knight Riders,0.667
32,336015,Sawai Mansingh Stadium,Rajasthan Royals,Rajasthan Royals,0.500
33,336016,"Punjab Cricket Association Stadium, Mohali",Royal Challengers Bengaluru,Punjab Kings,0.500
34,336017,Eden Gardens,Kolkata Knight Riders,Kolkata Knight Riders,0.333


In [9]:
home_venue_map = {
    "Wankhede Stadium": "Mumbai Indians",
    "MA Chidambaram Stadium": "Chennai Super Kings",
    "M Chinnaswamy Stadium": "Royal Challengers Bengaluru",
    "Eden Gardens": "Kolkata Knight Riders",
    "Arun Jaitley Stadium": "Delhi Capitals",
    "Rajiv Gandhi International Stadium": "Sunrisers Hyderabad",
    "Sawai Mansingh Stadium": "Rajasthan Royals",
    "Narendra Modi Stadium, Ahmedabad": "Gujarat Titans",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Kings",
    "Ekana Cricket Stadium": "Lucknow Super Giants",
}

matches_df["venue_home_team"] = matches_df["venue"].map(home_venue_map)
matches_df["team1_is_home"] = (matches_df["bat_first_team"] == matches_df["venue_home_team"]).astype(int)
matches_df["team2_is_home"] = (matches_df["bat_second_team"] == matches_df["venue_home_team"]).astype(int)

# what's the actual home win rate?
home_matches = matches_df[matches_df["venue_home_team"].notna() & (matches_df["result"] == "normal")].copy()
home_matches["home_team_won"] = (home_matches["venue_home_team"] == home_matches["winner"]).astype(int)
print("Overall home win rate:", home_matches["home_team_won"].mean().round(3))
print(home_matches.groupby("venue_home_team")["home_team_won"].mean().round(3).sort_values(ascending=False))

Overall home win rate: 0.481
venue_home_team
Rajasthan Royals               0.574
Chennai Super Kings            0.573
Kolkata Knight Riders          0.543
Sunrisers Hyderabad            0.511
Royal Challengers Bengaluru    0.490
Punjab Kings                   0.474
Mumbai Indians                 0.443
Lucknow Super Giants           0.407
Delhi Capitals                 0.376
Gujarat Titans                 0.346
Name: home_team_won, dtype: float64


In [10]:
matches_df.to_csv("../data/processed/matches.csv", index=False)
print("✅ Saved matches_df with all model features")
print("Shape:", matches_df.shape)
print("New feature columns:", [c for c in matches_df.columns if any(
    x in c for x in ["form", "h2h", "venue_avg", "venue_toss", "is_home"]
)])

✅ Saved matches_df with all model features
Shape: (1243, 55)
New feature columns: ['team1_form_runs_scored', 'team1_form_runs_conceded', 'team1_form_win_rate', 'team2_form_runs_scored', 'team2_form_runs_conceded', 'team2_form_win_rate', 'team1_h2h_win_rate', 'team1_h2h_matches', 'venue_avg_score', 'venue_toss_win_rate', 'team1_is_home', 'team2_is_home']


In [11]:
current_teams = [
    "Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru",
    "Kolkata Knight Riders", "Delhi Capitals", "Punjab Kings",
    "Rajasthan Royals", "Sunrisers Hyderabad", "Gujarat Titans", "Lucknow Super Giants"
]

decided = matches_df[
    (matches_df["result"] == "normal") & 
    (matches_df["winner"].notna())
].copy()

# ── 1. Home vs Away win % per team ──────────────────────────────────
records = []
for team in current_teams:
    team_matches = decided[
        (decided["bat_first_team"] == team) | 
        (decided["bat_second_team"] == team)
    ].copy()
    team_matches["won"] = (team_matches["winner"] == team).astype(int)
    team_matches["is_home"] = (team_matches["venue_home_team"] == team).astype(int)
    
    home = team_matches[team_matches["is_home"] == 1]
    away = team_matches[team_matches["is_home"] == 0]
    
    records.append({
        "team": team,
        "home_win_pct": round(home["won"].mean() * 100, 1) if len(home) > 0 else None,
        "away_win_pct": round(away["won"].mean() * 100, 1) if len(away) > 0 else None,
        "home_matches": len(home),
        "away_matches": len(away),
    })

home_away_df = pd.DataFrame(records).set_index("team")
home_away_df["home_advantage"] = (home_away_df["home_win_pct"] - home_away_df["away_win_pct"]).round(1)
home_away_df.sort_values("home_advantage", ascending=False)

,home_win_pct,away_win_pct,home_matches,away_matches,home_advantage
team,,,,,
Chennai Super Kings,66.3,51.4,83,181,14.9
Rajasthan Royals,59.1,46.9,66,179,12.2
Kolkata Knight Riders,57.0,48.5,100,171,8.5
Sunrisers Hyderabad,52.3,44.1,86,195,8.2
Mumbai Indians,59.2,51.3,98,189,7.9
Punjab Kings,49.3,45.4,75,196,3.9
Royal Challengers Bengaluru,51.0,51.1,96,184,-0.1
Gujarat Titans,58.1,63.0,31,46,-4.9
Delhi Capitals,41.8,48.1,91,181,-6.3


In [12]:
# ── 2. Best venue per team (where they win most) ────────────────────
team_venue_wins = []
for team in current_teams:
    team_matches = decided[
        (decided["bat_first_team"] == team) |
        (decided["bat_second_team"] == team)
    ].copy()
    team_matches["won"] = (team_matches["winner"] == team).astype(int)
    
    venue_stats = (
        team_matches.groupby("venue")
        .agg(matches=("won","count"), wins=("won","sum"))
    )
    venue_stats["win_pct"] = (venue_stats["wins"] / venue_stats["matches"] * 100).round(1)
    venue_stats = venue_stats[venue_stats["matches"] >= 5]  # min 5 matches at venue
    
    if len(venue_stats) > 0:
        best = venue_stats.sort_values("win_pct", ascending=False).iloc[0]
        team_venue_wins.append({
            "team": team,
            "best_venue": best.name,
            "win_pct": best["win_pct"],
            "matches": int(best["matches"]),
            "wins": int(best["wins"])
        })

best_venues_df = pd.DataFrame(team_venue_wins).set_index("team")
best_venues_df.sort_values("win_pct", ascending=False)

,best_venue,win_pct,matches,wins
team,,,,
Chennai Super Kings,Maharashtra Cricket Association Stadium,83.3,6,5
Mumbai Indians,Brabourne Stadium,71.4,7,5
Punjab Kings,Sharjah Cricket Stadium,71.4,7,5
Rajasthan Royals,Sheikh Zayed Stadium,71.4,7,5
Royal Challengers Bengaluru,"Punjab Cricket Association Stadium, Mohali",70.0,10,7
Kolkata Knight Riders,Rajiv Gandhi International Stadium,70.0,10,7
Delhi Capitals,Shaheed Veer Narayan Singh International Stadium,66.7,6,4
Gujarat Titans,Wankhede Stadium,66.7,6,4
Sunrisers Hyderabad,Arun Jaitley Stadium,64.7,17,11


In [13]:
# ── 3. Toss advantage by venue (venues with 10+ matches) ────────────
decided["toss_winner_won"] = (decided["toss_winner"] == decided["winner"]).astype(int)

toss_by_venue = (
    decided.groupby("venue")
    .agg(matches=("toss_winner_won","count"), toss_win_pct=("toss_winner_won","mean"))
)
toss_by_venue["toss_win_pct"] = (toss_by_venue["toss_win_pct"] * 100).round(1)
toss_by_venue = toss_by_venue[toss_by_venue["matches"] >= 10].sort_values("toss_win_pct", ascending=False)
toss_by_venue

,matches,toss_win_pct
venue,,
SuperSport Park,12,66.7
Ekana Cricket Stadium,27,66.7
Maharashtra Cricket Association Stadium,22,63.6
Kingsmead,15,60.0
Brabourne Stadium,10,60.0
Sharjah Cricket Stadium,28,57.1
Subrata Roy Sahara Stadium,16,56.2
Sawai Mansingh Stadium,68,55.9
"Dr DY Patil Sports Academy, Mumbai",20,55.0


In [14]:
home_away_df.to_csv("../data/processed/home_away_stats.csv")
best_venues_df.to_csv("../data/processed/best_venues.csv")
toss_by_venue.to_csv("../data/processed/toss_by_venue.csv")

print("✅ Venue & team EDA saved")

✅ Venue & team EDA saved


In [15]:
 # reload deliveries fresh with all columns
deliveries_df = pd.read_csv("../data/processed/deliveries.csv")
print(deliveries_df.columns.tolist())

['match_id', 'innings', 'innings_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'is_wicket', 'is_four', 'is_six', 'is_dot', 'phase', 'has_impact_player_rule', 'season', 'position', 'role', 'bowl_type']


In [17]:
matches_df = pd.read_csv("../data/processed/matches.csv")

deliveries_df = deliveries_df.merge(
    matches_df[["match_id", "has_impact_player_rule", "season"]],
    on="match_id", how="left"
)

# save permanently
deliveries_df.to_csv("../data/processed/deliveries.csv", index=False)
print("✅ Saved. Columns:", deliveries_df.columns.tolist())

✅ Saved. Columns: ['match_id', 'innings', 'innings_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'is_wicket', 'is_four', 'is_six', 'is_dot', 'phase', 'has_impact_player_rule_x', 'season_x', 'position', 'role', 'bowl_type', 'has_impact_player_rule_y', 'season_y']


In [20]:
# drop all duplicate columns
cols_to_drop = [c for c in deliveries_df.columns if c.endswith("_x") or c.endswith("_y")]
deliveries_df = deliveries_df.drop(columns=cols_to_drop)

# add back cleanly
deliveries_df = deliveries_df.merge(
    matches_df[["match_id", "has_impact_player_rule", "season"]],
    on="match_id", how="left"
)

deliveries_df.to_csv("../data/processed/deliveries.csv", index=False)
print("✅ Clean columns:", deliveries_df.columns.tolist())

✅ Clean columns: ['match_id', 'innings', 'innings_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'is_wicket', 'is_four', 'is_six', 'is_dot', 'phase', 'position', 'role', 'bowl_type', 'has_impact_player_rule', 'season']


In [22]:
deliveries_df = pd.read_csv("../data/processed/deliveries.csv")
print(deliveries_df.columns.tolist())

['match_id', 'innings', 'innings_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'is_wicket', 'is_four', 'is_six', 'is_dot', 'phase', 'position', 'role', 'bowl_type', 'has_impact_player_rule', 'season']


In [30]:
openers = deliveries_df[deliveries_df["role"] == "opener"]

opener_stats = (
    openers.groupby(["batter", "has_impact_player_rule"])
    .agg(
        runs=("runs_batter", "sum"),
        balls=("ball", "count"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        innings=("match_id", "nunique"),
    )
    .reset_index()  # ← this moves has_impact_player_rule back to a regular column
)

opener_stats["strike_rate"] = (opener_stats["runs"] / opener_stats["balls"] * 100).round(1)
opener_stats["boundary_pct"] = ((opener_stats["fours"] + opener_stats["sixes"]) / opener_stats["balls"] * 100).round(1)
opener_stats["avg_runs"] = (opener_stats["runs"] / opener_stats["innings"]).round(1)

top_openers = (
    opener_stats[opener_stats["has_impact_player_rule"] == True]
    [lambda df: df["innings"] >= 20]
    .sort_values("strike_rate", ascending=False)
    .head(15)
    [["batter", "innings", "runs", "avg_runs", "strike_rate", "boundary_pct"]]
)

print("=== Top Openers (Impact Player Era, min 20 innings) ===")
print(top_openers.to_string(index=False))

=== Top Openers (Impact Player Era, min 20 innings) ===
         batter  innings  runs  avg_runs  strike_rate  boundary_pct
  V Suryavanshi       22  1024      46.5        217.4          37.4
  Priyansh Arya       31   909      29.3        185.1          30.5
Abhishek Sharma       45  1448      32.2        178.3          29.1
        PD Salt       35  1148      32.8        169.8          27.8
        TM Head       38  1297      34.1        169.8          28.8
   RD Rickelton       23   805      35.0        165.0          26.0
      SP Narine       24   661      27.5        164.4          28.1
 P Simran Singh       46  1492      32.4        160.1          25.3
       MR Marsh       30  1301      43.4        157.1          23.8
      SV Samson       24   858      35.8        155.4          24.6
    YBK Jaiswal       59  2046      34.7        152.7          25.1
   Shubman Gill       57  2538      44.5        151.3          20.3
   Ishan Kishan       32   940      29.4        150.4       

In [34]:
openers = deliveries_df[deliveries_df["role"] == "opener"].copy()

opener_stats = openers.groupby(["batter", "has_impact_player_rule"]).agg(
    runs=("runs_batter", "sum"),
    balls=("ball", "count"),
    fours=("is_four", "sum"),
    sixes=("is_six", "sum"),
    innings=("match_id", "nunique"),
).reset_index()

opener_stats["strike_rate"] = (opener_stats["runs"] / opener_stats["balls"] * 100).round(1)
opener_stats["boundary_pct"] = ((opener_stats["fours"] + opener_stats["sixes"]) / opener_stats["balls"] * 100).round(1)
opener_stats["avg_runs"] = (opener_stats["runs"] / opener_stats["innings"]).round(1)

# filter step by step — no chaining
top_openers = opener_stats[opener_stats["has_impact_player_rule"] == True]
top_openers = top_openers[top_openers["innings"] >= 20]
top_openers = top_openers.sort_values("strike_rate", ascending=False)
top_openers = top_openers.head(15)
top_openers = top_openers[["batter","innings","runs","avg_runs","strike_rate","boundary_pct"]]

print("=== Top Openers (Impact Player Era, min 20 innings) ===")
print(top_openers.to_string(index=False))

=== Top Openers (Impact Player Era, min 20 innings) ===
         batter  innings  runs  avg_runs  strike_rate  boundary_pct
  V Suryavanshi       22  1024      46.5        217.4          37.4
  Priyansh Arya       31   909      29.3        185.1          30.5
Abhishek Sharma       45  1448      32.2        178.3          29.1
        PD Salt       35  1148      32.8        169.8          27.8
        TM Head       38  1297      34.1        169.8          28.8
   RD Rickelton       23   805      35.0        165.0          26.0
      SP Narine       24   661      27.5        164.4          28.1
 P Simran Singh       46  1492      32.4        160.1          25.3
       MR Marsh       30  1301      43.4        157.1          23.8
      SV Samson       24   858      35.8        155.4          24.6
    YBK Jaiswal       59  2046      34.7        152.7          25.1
   Shubman Gill       57  2538      44.5        151.3          20.3
   Ishan Kishan       32   940      29.4        150.4       

In [35]:
# ── powerplay bowlers ────────────────────────────────────────────────
pp_bowlers = deliveries_df[deliveries_df["phase"] == "powerplay"].copy()

pp_bowling = pp_bowlers.groupby(["bowler", "has_impact_player_rule"]).agg(
    runs=("runs_total", "sum"),
    balls=("ball", "count"),
    wickets=("is_wicket", "sum"),
).reset_index()

pp_bowling["economy"] = (pp_bowling["runs"] / pp_bowling["balls"] * 6).round(2)
pp_bowling["strike_rate"] = (pp_bowling["balls"] / pp_bowling["wickets"].replace(0, np.nan)).round(1)

top_pp = pp_bowling[pp_bowling["has_impact_player_rule"] == True]
top_pp = top_pp[top_pp["balls"] >= 200]
top_pp = top_pp.sort_values("economy")
top_pp = top_pp.head(15)
top_pp = top_pp[["bowler", "balls", "wickets", "economy", "strike_rate"]]

print("=== Best Powerplay Bowlers (Impact Player Era, min 200 balls) ===")
print(top_pp.to_string(index=False))

# ── death over bowlers ───────────────────────────────────────────────
death_bowlers = deliveries_df[deliveries_df["phase"] == "death"].copy()

death_bowling = death_bowlers.groupby(["bowler", "has_impact_player_rule"]).agg(
    runs=("runs_total", "sum"),
    balls=("ball", "count"),
    wickets=("is_wicket", "sum"),
).reset_index()

death_bowling["economy"] = (death_bowling["runs"] / death_bowling["balls"] * 6).round(2)
death_bowling["strike_rate"] = (death_bowling["balls"] / death_bowling["wickets"].replace(0, np.nan)).round(1)

top_death = death_bowling[death_bowling["has_impact_player_rule"] == True]
top_death = top_death[top_death["balls"] >= 150]
top_death = top_death.sort_values("economy")
top_death = top_death.head(15)
top_death = top_death[["bowler", "balls", "wickets", "economy", "strike_rate"]]

print("\n=== Best Death Over Bowlers (Impact Player Era, min 150 balls) ===")
print(top_death.to_string(index=False))

=== Best Powerplay Bowlers (Impact Player Era, min 200 balls) ===
        bowler  balls  wickets  economy  strike_rate
     JJ Bumrah    381       13     7.39         29.3
      TA Boult    633       32     7.76         19.8
       B Kumar    817       42     7.76         19.5
Sandeep Sharma    381       12     7.89         31.8
Mohammed Siraj    912       40     7.91         22.8
Mohammed Shami    608       31     7.99         19.6
      KK Ahmed    621       22     8.26         28.2
     DL Chahar    612       29     8.27         21.1
   T Natarajan    202        6     8.47         33.7
  JR Hazlewood    331       19     8.61         17.4
  M Theekshana    229        4     8.62         57.2
  TU Deshpande    472       17     8.63         27.8
      I Sharma    276       15     8.70         18.4
     SM Curran    252        3     8.71         84.0
      AR Patel    268        9     8.71         29.8

=== Best Death Over Bowlers (Impact Player Era, min 150 balls) ===
        bowler  ba

In [36]:
top_openers.to_csv("../data/processed/top_openers.csv", index=False)
top_pp.to_csv("../data/processed/top_pp_bowlers.csv", index=False)
top_death.to_csv("../data/processed/top_death_bowlers.csv", index=False)

print("✅ Player stats saved")

✅ Player stats saved


In [33]:
print(deliveries_df.dtypes)


match_id                    int64
innings                     int64
innings_team                  str
over                        int64
ball                      float64
batter                        str
bowler                        str
non_striker                   str
runs_batter                 int64
runs_extras                 int64
runs_total                  int64
is_wicket                    bool
is_four                      bool
is_six                       bool
is_dot                       bool
phase                         str
position                    int64
role                          str
bowl_type                     str
has_impact_player_rule       bool
season                      int64
dtype: object


In [32]:
print(deliveries_df["has_impact_player_rule"].value_counts())

has_impact_player_rule
False    225954
True      69778
Name: count, dtype: int64


In [25]:
# ── 2. Best powerplay bowlers (min 200 balls in powerplay) ───────────
pp_bowlers = deliveries_df[deliveries_df["phase"] == "powerplay"]

pp_bowling = (
    pp_bowlers.groupby(["bowler", "has_impact_player_rule"])
    .agg(
        runs=("runs_total", "sum"),
        balls=("ball", "count"),
        wickets=("is_wicket", "sum"),
    )
)
pp_bowling["economy"] = (pp_bowling["runs"] / pp_bowling["balls"] * 6).round(2)
pp_bowling["strike_rate"] = (pp_bowling["balls"] / pp_bowling["wickets"].replace(0, np.nan)).round(1)

top_pp_bowlers = (
    pp_bowling[pp_bowling["has_impact_player_rule"] == True]
    .reset_index()
    [lambda df: df["balls"] >= 200]
    .sort_values("economy")
    .head(15)
    [["bowler", "balls", "wickets", "economy", "strike_rate"]]
)

print("=== Best Powerplay Bowlers (Impact Player Era, min 200 balls) ===")
print(top_pp_bowlers.to_string(index=False))

KeyError: 'has_impact_player_rule'

In [ ]:
# ── 3. Best death over specialists (min 150 balls in death overs) ────
death_bowlers = deliveries_df[deliveries_df["phase"] == "death"]

death_bowling = (
    death_bowlers.groupby(["bowler", "has_impact_player_rule"])
    .agg(
        runs=("runs_total", "sum"),
        balls=("ball", "count"),
        wickets=("is_wicket", "sum"),
    )
)
death_bowling["economy"] = (death_bowling["runs"] / death_bowling["balls"] * 6).round(2)
death_bowling["strike_rate"] = (death_bowling["balls"] / death_bowling["wickets"].replace(0, np.nan)).round(1)

top_death_bowlers = (
    death_bowling[death_bowling["has_impact_player_rule"] == True]
    .reset_index()
    [lambda df: df["balls"] >= 150]
    .sort_values("economy")
    .head(15)
    [["bowler", "balls", "wickets", "economy", "strike_rate"]]
)

print("=== Best Death Over Bowlers (Impact Player Era, min 150 balls) ===")
print(top_death_bowlers.to_string(index=False))

In [37]:
import json, math

def clean_nans(obj):
    if isinstance(obj, dict):
        return {k: clean_nans(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_nans(v) for v in obj]
    elif isinstance(obj, float) and math.isnan(obj):
        return None
    else:
        return obj

# reload all stats
home_away_df = pd.read_csv("../data/processed/home_away_stats.csv")
best_venues_df = pd.read_csv("../data/processed/best_venues.csv")
toss_by_venue = pd.read_csv("../data/processed/toss_by_venue.csv")
top_openers = pd.read_csv("../data/processed/top_openers.csv")
top_pp = pd.read_csv("../data/processed/top_pp_bowlers.csv")
top_death = pd.read_csv("../data/processed/top_death_bowlers.csv")

stats_data = {
    "home_away": home_away_df.to_dict(orient="records"),
    "best_venues": best_venues_df.to_dict(orient="records"),
    "toss_by_venue": toss_by_venue.to_dict(orient="records"),
    "top_openers": top_openers.to_dict(orient="records"),
    "top_pp_bowlers": top_pp.to_dict(orient="records"),
    "top_death_bowlers": top_death.to_dict(orient="records"),
}

stats_data = clean_nans(stats_data)

with open("../website/stats_data.json", "w") as f:
    json.dump(stats_data, f, indent=2)

print("✅ stats_data.json saved")

✅ stats_data.json saved


In [38]:
pp_wickets = (
    deliveries_df[deliveries_df["phase"] == "powerplay"]
    .groupby(["match_id", "innings"])["is_wicket"]
    .sum()
    .reset_index()
)

pp_wickets_inn1 = pp_wickets[pp_wickets["innings"] == 1][["match_id", "is_wicket"]].rename(
    columns={"is_wicket": "pp_wickets_inn1"}
)

matches_df = matches_df.merge(pp_wickets_inn1, on="match_id", how="left")
matches_df["pp_run_rate_inn1"] = (matches_df["pp_runs_inn1"] / 6).round(2)

matches_df[["match_id", "pp_runs_inn1", "pp_wickets_inn1", "pp_run_rate_inn1"]].head()

,match_id,pp_runs_inn1,pp_wickets_inn1,pp_run_rate_inn1
0,335982,61.0,1,10.17
1,335983,53.0,1,8.83
2,335984,40.0,2,6.67
3,335986,39.0,2,6.50
4,335985,47.0,3,7.83


In [39]:
matches_df.to_csv("../data/processed/matches.csv", index=False)
print("✅ Saved. Shape:", matches_df.shape)

✅ Saved. Shape: (1243, 57)


In [40]:
matches_df.to_csv("../data/processed/matches.csv", index=False)
print("✅ Saved. Shape:", matches_df.shape)

✅ Saved. Shape: (1243, 57)


In [41]:
matches_df = pd.read_csv("../data/processed/matches.csv")
matches_df["date"] = pd.to_datetime(matches_df["date"])
deliveries_df = pd.read_csv("../data/processed/deliveries.csv")

# rebuild team_match_log
team_match_log = pd.concat([
    matches_df[["match_id","date","bat_first_team","bat_second_team",
                "total_runs_inn1","total_runs_inn2","winner"]].rename(
        columns={"bat_first_team":"team","bat_second_team":"opponent",
                 "total_runs_inn1":"runs_scored","total_runs_inn2":"runs_conceded"}),
    matches_df[["match_id","date","bat_second_team","bat_first_team",
                "total_runs_inn2","total_runs_inn1","winner"]].rename(
        columns={"bat_second_team":"team","bat_first_team":"opponent",
                 "total_runs_inn2":"runs_scored","total_runs_inn1":"runs_conceded"})
], ignore_index=True)
team_match_log["won"] = (team_match_log["team"] == team_match_log["winner"]).astype(int)
team_match_log = team_match_log.sort_values(["team","date"]).reset_index(drop=True)

print("✅ Base data loaded")
print("matches_df:", matches_df.shape)
print("deliveries_df:", deliveries_df.shape)
print("team_match_log:", team_match_log.shape)

✅ Base data loaded
matches_df: (1243, 57)
deliveries_df: (295732, 21)
team_match_log: (2486, 8)


In [42]:
# cumulative win rate within each season, shifted to avoid leakage
team_match_log["season"] = pd.to_datetime(team_match_log["date"]).dt.year

team_match_log["season_wins_so_far"] = (
    team_match_log.groupby(["team","season"])["won"]
    .transform(lambda x: x.shift(1).expanding().sum())
)
team_match_log["season_matches_so_far"] = (
    team_match_log.groupby(["team","season"])["won"]
    .transform(lambda x: x.shift(1).expanding().count())
)
team_match_log["season_win_rate"] = (
    team_match_log["season_wins_so_far"] /
    team_match_log["season_matches_so_far"]
).round(3)

# merge onto matches_df for both teams
bat_first_swr = team_match_log[["match_id","team","season_win_rate"]].rename(
    columns={"team":"bat_first_team","season_win_rate":"team1_season_win_rate"})
bat_second_swr = team_match_log[["match_id","team","season_win_rate"]].rename(
    columns={"team":"bat_second_team","season_win_rate":"team2_season_win_rate"})

matches_df = matches_df.merge(bat_first_swr, on=["match_id","bat_first_team"], how="left")
matches_df = matches_df.merge(bat_second_swr, on=["match_id","bat_second_team"], how="left")

print("Season win rate sample:")
print(matches_df[["match_id","bat_first_team","team1_season_win_rate",
                   "bat_second_team","team2_season_win_rate"]].dropna().head(5))

Season win rate sample:
   match_id       bat_first_team  team1_season_win_rate   bat_second_team  \
5    335987         Punjab Kings                    0.0  Rajasthan Royals   
6    335988  Sunrisers Hyderabad                    0.0    Delhi Capitals   
7    335989  Chennai Super Kings                    1.0    Mumbai Indians   
8    335990  Sunrisers Hyderabad                    0.0  Rajasthan Royals   
9    335991         Punjab Kings                    0.0    Mumbai Indians   

   team2_season_win_rate  
5                    0.0  
6                    1.0  
7                    0.0  
8                    0.5  
9                    0.0  


In [43]:
# for each match, what has THIS team averaged at THIS venue in prior matches
team_venue_avg = []

matches_sorted = matches_df.sort_values("date").reset_index(drop=True)

for idx, row in matches_sorted.iterrows():
    # batting team's prior scores at this venue
    prior = team_match_log[
        (team_match_log["team"] == row["bat_first_team"]) &
        (team_match_log["date"] < row["date"])
    ]
    # need to also know which matches were at this venue
    prior_venue = matches_df[
        (matches_df["venue"] == row["venue"]) &
        (matches_df["date"] < row["date"]) &
        ((matches_df["bat_first_team"] == row["bat_first_team"]) |
         (matches_df["bat_second_team"] == row["bat_first_team"]))
    ]
    if len(prior_venue) >= 3:
        # get runs scored by this team at this venue
        inn1_scores = prior_venue[prior_venue["bat_first_team"] == row["bat_first_team"]]["total_runs_inn1"]
        inn2_scores = prior_venue[prior_venue["bat_second_team"] == row["bat_first_team"]]["total_runs_inn2"]
        all_scores = pd.concat([inn1_scores, inn2_scores])
        team_venue_avg.append(all_scores.mean().round(1) if len(all_scores) > 0 else np.nan)
    else:
        team_venue_avg.append(np.nan)

matches_sorted["team1_venue_avg"] = team_venue_avg
matches_df = matches_sorted.copy()

print("Team venue avg sample:")
print(matches_df[["bat_first_team","venue","team1_venue_avg"]].dropna().head(8))

Team venue avg sample:
                 bat_first_team                                       venue  \
20                 Punjab Kings  Punjab Cricket Association Stadium, Mohali   
21  Royal Challengers Bengaluru                       M Chinnaswamy Stadium   
24  Royal Challengers Bengaluru                       M Chinnaswamy Stadium   
25          Chennai Super Kings                      MA Chidambaram Stadium   
30          Chennai Super Kings                      MA Chidambaram Stadium   
34        Kolkata Knight Riders                                Eden Gardens   
36               Delhi Capitals                        Arun Jaitley Stadium   
38             Rajasthan Royals                      Sawai Mansingh Stadium   

    team1_venue_avg  
20            183.7  
21            127.3  
24            134.5  
25            176.3  
30            168.2  
34            126.0  
36            170.0  
38            154.8  


In [44]:
# proxy: bowling team's economy rate conceded in last 5 matches
# lower economy = stronger bowling attack
team_match_log["bowling_economy"] = (
    team_match_log["runs_conceded"] / 120 * 6
).round(2)  # approximate economy (120 balls per innings)

team_match_log["opp_bowling_strength"] = (
    team_match_log.groupby("team")["bowling_economy"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .round(2)
)

# merge as bowling team's strength
bat_first_opp = team_match_log[["match_id","team","opp_bowling_strength"]].rename(
    columns={"team":"bat_second_team",
             "opp_bowling_strength":"team2_bowling_strength"})
bat_second_opp = team_match_log[["match_id","team","opp_bowling_strength"]].rename(
    columns={"team":"bat_first_team",
             "opp_bowling_strength":"team1_bowling_strength"})

matches_df = matches_df.merge(bat_first_opp, on=["match_id","bat_second_team"], how="left")
matches_df = matches_df.merge(bat_second_opp, on=["match_id","bat_first_team"], how="left")

print("Bowling strength sample:")
print(matches_df[["bat_first_team","bat_second_team",
                   "team1_bowling_strength","team2_bowling_strength"]].dropna().head(5))

Bowling strength sample:
        bat_first_team   bat_second_team  team1_bowling_strength  \
5         Punjab Kings  Rajasthan Royals                   12.00   
6  Sunrisers Hyderabad    Delhi Capitals                    5.60   
7  Chennai Super Kings    Mumbai Indians                   10.35   
8  Sunrisers Hyderabad  Rajasthan Royals                    6.38   
9         Punjab Kings    Mumbai Indians                   10.20   

   team2_bowling_strength  
5                    6.60  
6                    6.45  
7                    8.30  
8                    7.45  
9                    9.35  


In [45]:
# for each team, their strike rate against spin and pace historically
# use deliveries_df which has bowl_type

batting_vs_type = (
    deliveries_df.dropna(subset=["bowl_type"])
    .groupby(["innings_team", "bowl_type"])
    .agg(runs=("runs_batter","sum"), balls=("ball","count"))
    .reset_index()
)
batting_vs_type["sr_vs_type"] = (batting_vs_type["runs"] / batting_vs_type["balls"] * 100).round(2)

spin_sr = batting_vs_type[batting_vs_type["bowl_type"]=="spin"][["innings_team","sr_vs_type"]].rename(
    columns={"innings_team":"team","sr_vs_type":"team_sr_vs_spin"})
pace_sr = batting_vs_type[batting_vs_type["bowl_type"]=="pace"][["innings_team","sr_vs_type"]].rename(
    columns={"innings_team":"team","sr_vs_type":"team_sr_vs_pace"})

# merge for batting team
matches_df = matches_df.merge(
    spin_sr.rename(columns={"team":"bat_first_team","team_sr_vs_spin":"team1_sr_vs_spin"}),
    on="bat_first_team", how="left"
)
matches_df = matches_df.merge(
    pace_sr.rename(columns={"team":"bat_first_team","team_sr_vs_pace":"team1_sr_vs_pace"}),
    on="bat_first_team", how="left"
)

print("Batting vs type sample:")
print(matches_df[["bat_first_team","team1_sr_vs_spin","team1_sr_vs_pace"]].dropna().head(5))

Batting vs type sample:
          bat_first_team  team1_sr_vs_spin  team1_sr_vs_pace
0  Kolkata Knight Riders            119.35            125.51
1    Chennai Super Kings            116.51            127.69
2       Rajasthan Royals            118.95            127.13
3    Sunrisers Hyderabad            121.98            125.56
4         Mumbai Indians            121.91            128.25


In [46]:
matches_df.to_csv("../data/processed/matches.csv", index=False)
print("✅ Saved. Shape:", matches_df.shape)
print("New features:", [c for c in matches_df.columns if any(
    x in c for x in ["season_win","venue_avg","bowling_strength","sr_vs","h2h_venue"]
)])

✅ Saved. Shape: (1243, 64)
New features: ['venue_avg_score', 'team1_season_win_rate', 'team2_season_win_rate', 'team1_venue_avg', 'team2_bowling_strength', 'team1_bowling_strength', 'team1_sr_vs_spin', 'team1_sr_vs_pace']
